In [34]:
import numpy as np # importa la librería usada para trabajar con vetores y matrices
import pandas as pd # importa la librería usada para trabajar con dataframes

Los datos Colombianos generalmente suelen venir en Latin-1, pero Pandas por defecto usa UTF-8 que es el codificador mas usado. Asi que especificamos el encoder que necesitamos.

In [35]:
#Importar el dataset y guardarlo en un dataframe
df_mortalidad_bgta = pd.read_csv('../raw/mortalidad_bogota.csv',sep=';', index_col=None, encoding='latin-1')
display(df_mortalidad_bgta.head(n=10))

,Año,Código,Localidad,Casos,Población,Tasa X 100.000 habitantes
0,2017,1,Usaquén,22.00,528924,"4,2"
1,2017,2,Chapinero,15.00,154506,"9,7"
2,2017,3,Santa Fe,9.00,102984,"8,7"
3,2017,4,San Cristóbal,24.00,382696,"6,3"
4,2017,5,Usme,22.00,358813,"6,1"
5,2017,6,Tunjuelito,29.00,170440,17
6,2017,7,Bosa,31.00,684570,"4,5"
7,2017,8,Kennedy,88.00,1009665,"8,7"
8,2017,9,Fontibón,36.00,361474,10
9,2017,10,Engativá,42.00,786139,"5,3"


## 1. Eliminacion de columnas no necesarias 
En este caso dejaremos todas las columnas, incluyendo el codigo. Ya que nos servira para visualizar los registros que tenian faltantes despues de tratarlos. Ya que los idx cambiarian si eliminamos algunos.

## 2. Conversion de las columnas numericas al estandar aglosajon
Viendo el dataset tiene usa para los miles puntos (.), y comas (,) para los decimales, que hace referencia a la notacion que usamos en Colombia. Sin embargo para Python y Pandas para los miles o millones no usan nada, solo para los decimales las puntos. Por lo cual lo limpiaremos para las columnas 'Poblacion'.

In [36]:
df_mortalidad_bgta['Población'] = (df_mortalidad_bgta['Población']
    .str.replace('.', '', regex=False)   # quita punto de miles o millones
    .str.replace(',', '.', regex=False)  # cambia coma decimal por punto
)

df_mortalidad_bgta['Tasa X 100.000 habitantes'] = (df_mortalidad_bgta['Tasa X 100.000 habitantes']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)

## 3. Numeros guardados como texto
Identificamos que 'Tasa X 100.000 habitantes', 'Poblacion' y 'Casos 'fueron almacenados como texto, sin embargo es de nuestro interes tenerlos numericos para poder verificar datos atipicos.

In [37]:
print(df_mortalidad_bgta.dtypes)

Año                            int64
Código                         int64
Localidad                        str
Casos                        float64
Población                        str
Tasa X 100.000 habitantes        str
dtype: object


## 3.1 Tratamiento de Numeros guardados como texto en la Tasa y en 'Poblacion'
Los convertiremos a numericos, mas especificamente a flotantes y si no se puede se pone un NaN, por lo cual verificaremos nuevamente si hay faltantes

In [38]:
df_mortalidad_bgta['Población'] = pd.to_numeric(df_mortalidad_bgta['Población'], errors='coerce')
df_mortalidad_bgta['Tasa X 100.000 habitantes'] = pd.to_numeric(df_mortalidad_bgta['Tasa X 100.000 habitantes'], errors='coerce')
df_mortalidad_bgta['Casos'] = pd.to_numeric(df_mortalidad_bgta['Casos'], errors='coerce')

print(df_mortalidad_bgta.dtypes)

Año                            int64
Código                         int64
Localidad                        str
Casos                        float64
Población                    float64
Tasa X 100.000 habitantes    float64
dtype: object


## 4. Buscaremos datos faltantes 

Los almacenaremos en una variable e imprimimos solo los registros que tengan datos faltantes.

In [39]:
idxRowNan = pd.isnull(df_mortalidad_bgta).any(axis=1).to_numpy().nonzero()
display(df_mortalidad_bgta.iloc[idxRowNan])

# Guardamos los codigos para visualizar despues de tratarlos
codigos_afectados = df_mortalidad_bgta.loc[
    pd.isnull(df_mortalidad_bgta).any(axis=1), 'Código'
].unique()

,Año,Código,Localidad,Casos,Población,Tasa X 100.000 habitantes
16,2017,17,La Candelaria,NaN,17049.00,0.00
86,2020,99,Sin dato,14.00,NaN,0.00
108,2021,21,Sin dato,17.00,NaN,0.00
129,2022,20,Sumapaz,NaN,3530.00,0.00
130,2022,99,Sin dato,49.00,NaN,0.00
151,2023,20,Sumapaz,NaN,3574.00,0.00
152,2023,99,Sin dato,53.00,NaN,0.00
195,2015,20,Sumapaz,NaN,3224.00,0.00
218,2024,99,Sin dato,51.00,NaN,NaN
236,2025,17,La Candelaria,NaN,16801.00,0.00


Revisemos los codigos...

In [40]:
print("Codigos de las localidades que tienen vacios: ", codigos_afectados)

Codigos de las localidades que tienen vacios:  [17 99 21 20]


## 4.1 Tratamiento a datos faltantes en 'Localidad'
Identificamos que el codigo 17 hace referencia a la localidad de La Candelaria y el 20 a Sumapaz, pero el 21 y 99 a ninguna. Ademas no hace sentido ya que son 20 localidades.

In [41]:
display(df_mortalidad_bgta[df_mortalidad_bgta['Código'].isin(codigos_afectados)][['Código', 'Localidad']].drop_duplicates().sort_values('Código'))

,Código,Localidad
16,17,La Candelaria
19,20,Sumapaz
108,21,Sin dato
20,99,Sin dato


Para solucionar eso **eliminaremos los registros de codigos de 99, 21 y 0**

In [42]:
idx_eliminar = df_mortalidad_bgta[df_mortalidad_bgta['Código'].isin([99, 21, 0])].index
df_mortalidad_bgta = df_mortalidad_bgta.drop(idx_eliminar)

display(df_mortalidad_bgta[df_mortalidad_bgta['Código'].isin(codigos_afectados)][['Código', 'Localidad']].drop_duplicates().sort_values('Código'))

,Código,Localidad
16,17,La Candelaria
19,20,Sumapaz


## 4.2 Tratamiento de valores faltantes en columna 'Casos' (Pendiente por revisar)

En la columna **Casos**, algunos registros con NaN, lo cual se interpreta como ausencia de muertes reportadas ese año en esa localidad, por lo que lo reemplazamos con 0. 

In [43]:
idx_casos_nan = df_mortalidad_bgta[df_mortalidad_bgta['Casos'].isna()].index
df_mortalidad_bgta['Casos'] = df_mortalidad_bgta['Casos'].fillna(0)

display(df_mortalidad_bgta.loc[idx_casos_nan])

,Año,Código,Localidad,Casos,Población,Tasa X 100.000 habitantes
16,2017,17,La Candelaria,0.00,17049.00,0.00
129,2022,20,Sumapaz,0.00,3530.00,0.00
151,2023,20,Sumapaz,0.00,3574.00,0.00
195,2015,20,Sumapaz,0.00,3224.00,0.00
236,2025,17,La Candelaria,0.00,16801.00,0.00


## 4.3 Tratamiento de valores faltantes en columna 'Tasa x 100 habitantes'
La columna **Tasa x 100.000 habitantes** presenta NaN, pero comprobamos solo estaba en los registros de las localidades inexistentes

In [44]:
idx_tasa_nan = df_mortalidad_bgta[df_mortalidad_bgta['Tasa X 100.000 habitantes'].isna()].index

if(len(idx_tasa_nan > 0)):
    print("Indices de faltantes en la tasa: ", idx_tasa_nan)
else:
    print("No hay registros duplicados")
display(df_mortalidad_bgta.loc[idx_tasa_nan])

No hay registros duplicados


,Año,Código,Localidad,Casos,Población,Tasa X 100.000 habitantes


## 4.4 Revision de faltantes despues de tratarlos
Comprobamos que efectivamente ya no tenemos datos faltantes.

In [45]:
print(df_mortalidad_bgta[['Población', 'Tasa X 100.000 habitantes', 'Casos']].isna().sum())

Población                    0
Tasa X 100.000 habitantes    0
Casos                        0
dtype: int64


## 5. Registros duplicados
No hay registros duplicados, por lo cual vamos bien.

In [46]:
display(df_mortalidad_bgta[df_mortalidad_bgta.duplicated()])

,Año,Código,Localidad,Casos,Población,Tasa X 100.000 habitantes


## 6. Normalizacion de texto 
Comprobemos que todas las localidades estan escritas de igual manera

In [47]:
print(df_mortalidad_bgta['Localidad'].unique())

<StringArray>
[           'Usaquén',          'Chapinero',           'Santa Fe',
      'San Cristóbal',               'Usme',         'Tunjuelito',
               'Bosa',            'Kennedy',           'Fontibón',
           'Engativá',               'Suba',     'Barrios Unidos',
        'Teusaquillo',       'Los Mártires',     'Antonio Nariño',
      'Puente Aranda',      'La Candelaria', 'Rafael Uribe Uribe',
     'Ciudad Bolívar',            'Sumapaz']
Length: 20, dtype: str


## 7. Datos atipicos
Revisemos si hay algun dato atipico en las columnas Casos, Poblacion o en la Tasa

In [48]:
pd.set_option('display.float_format', '{:.2f}'.format)
df_detalles = df_mortalidad_bgta[['Casos', 'Población', 'Tasa X 100.000 habitantes']].describe()
display(df_detalles)

,Casos,Población,Tasa X 100.000 habitantes
count,220.00,220.00,220.00
mean,27.09,382366.35,10.52
std,17.72,336493.96,8.32
min,0.00,3137.00,0.00
25%,17.00,128191.25,5.57
50%,24.00,303027.50,7.70
75%,36.25,581124.00,13.50
max,88.00,1246637.00,60.60


Notemos que los datos estan dentro de un rango logico y aparantemente seguros para poder trabajar con ellos.

## Por ultimo guardemos el dataset

In [50]:
df_mortalidad_bgta.to_csv('../datasets_limpios/mortalidad_bogota_limpio.csv', sep=';', index=False, encoding='utf-8')